In [ ]:
import os
import glob
from skimage import io

import torch
import matplotlib.pyplot as plt
import numpy as np
from skimage.color import label2rgb
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
import monai
from transforms import (
    transforms_fn,
    visualize_augmentations,
)
from utils import (
    load_checkpoint,
    save_checkpoint,
    model_fn,
    get_datasets,
    get_loaders,
    evaluate_fn,
    show_images,
    loss_plot_fn,
    loss_plot_log_fn,
    plot_segmentation_scores_fn,
)

from create_bioimageio_model import export_bioimageio

In [ ]:
# Check GPU access
print(torch.version.cuda)
if torch.cuda.is_available():
    print("GPU available")
    print(f"Total Memory: {torch.cuda.get_device_properties(0).total_memory}")
else:
    print("No GPU")

!nvidia-smi

In [ ]:
# Hyperparameters
LEARNING_RATE = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 2
NUM_EPOCHS = 10
NUM_WORKERS = 2
IMAGE_HEIGHT = 512
IMAGE_WIDTH = 512
PIN_MEMORY = True
LOAD_MODEL = True
TRAIN_IMG_DIR = "data_tem/train_images/" # data_tem, data_brightfield
TRAIN_SEM_DIR = "data_tem/train_masks/"
TRAIN_MASK_DIR = "data_tem/train_labels/"
VAL_IMG_DIR = "data_tem/val_images/"
VAL_SEM_DIR = "data_tem/val_masks/"
VAL_MASK_DIR = "data_tem/val_labels/"
SHOW_VAL_INTERVAL = 10
BIOIMAGEIO = "best" # "none", "best", "last"
DEBUG = False
TEST=False

In [ ]:
def train_fn(loader, model, optimizer, loss_fn, scaler):
    loop = tqdm(loader)
    train_loss_all= []

    for batch_index, (data, targets) in enumerate(loop):
        # data to device
        #print(data[1].shape)
        #plt.imshow(data[1])
        data = data.to(device=DEVICE)
        # targets to device
        targets = np.stack(targets, axis=1)
        targets = torch.tensor(targets).to(device=DEVICE)

        # forward
        with torch.cuda.amp.autocast():
            predictions = model(data)
            predictions = predictions.to(torch.float)  # Convert predictions to float
            #print(predictions.shape)  # Check shape of predictions tensor
            #print(targets.shape) 
            train_crossentropy_loss = loss_fn[0](predictions[:,0:3,:,:], targets[:,2,:,:].long()) # semantic
            train_mse_loss1 = loss_fn[1](predictions[:,3,:,:], targets[:,3,:,:].float()) # fibre DT
            train_mse_loss2 = loss_fn[1](predictions[:,4,:,:], targets[:,4,:,:].float()) # axon DT

            train_crossentropy_loss = train_crossentropy_loss.float()
            train_mse_loss1 = train_mse_loss1.float()
            train_mse_loss2 = train_mse_loss2.float()
            train_mse_loss = train_mse_loss1 + train_mse_loss2
            train_loss = torch.add(train_crossentropy_loss, train_mse_loss)            
    
        # backward
        optimizer.zero_grad()
        scaler.scale(train_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # update tqdm loop
        loop.set_postfix(train_loss=train_loss.item())
        train_loss_all.append(train_loss.item())
    
    return sum(train_loss_all)/len(train_loss_all)

In [ ]:
def main():
    # get datasets and do image cache
    train_transform, val_transform = transforms_fn(IMAGE_HEIGHT, IMAGE_WIDTH)

    train_dataset, val_dataset = get_datasets(
        TRAIN_IMG_DIR,
        TRAIN_SEM_DIR,
        TRAIN_MASK_DIR,
        VAL_IMG_DIR,
        VAL_SEM_DIR,
        VAL_MASK_DIR,
        train_transform,
        val_transform,
    )

    # Load image data in train dataset cache
    train_dataset.populate_cache()
    print(train_dataset)

    #image, masks = train_dataset.__getitem__(1)
    #print(f"Image shape: {image.shape}")
    #print(f"Number of masks: {len(masks)}")
    #show_images(
    #    image,
    #    masks[0],
    #    masks[1],
    #    masks[2],
    #    masks[3],
    #    titles = ["Image1", "Image2", "Image3", "Image4", "Image5"],
    #    n_cols = 3
    #)

    # Load image data in val dataset cache
    val_dataset.populate_cache()
    print(val_dataset)    

    model = model_fn(DEVICE)
    crossentropy_loss = nn.CrossEntropyLoss()
    mse_loss = nn.MSELoss()
    loss_fn = [crossentropy_loss, mse_loss]
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


    if LOAD_MODEL:
        last_epoch, train_loss, val_loss, f1_fibre, f1_axon, dice_score, balanced_segmentation_score, best_score = load_checkpoint(torch.load("model_checkpoint.pth.tar"), model, optimizer)

        train_loader, val_loader = get_loaders(
            train_dataset,
            val_dataset,
            BATCH_SIZE,
            NUM_WORKERS,
            PIN_MEMORY,
        )

        val_loss_load, f1_fibre_load, f1_axon_load, dice_score_load = evaluate_fn(val_loader, model, loss_fn, device=DEVICE, show_results=True)
        
        print("Model successfully loaded")
        print("Val loss: {:.4f}".format(val_loss_load))
        print("Fibre F1 score: {:.4f}".format(f1_fibre_load))
        print("Axon F1 score: {:.4f}".format(f1_axon_load))
        print("Dice score: {:.4f}".format(dice_score_load))
    else:
        train_loss = []
        val_loss = []
        f1_fibre = []
        f1_axon = []
        dice_score = []
        balanced_segmentation_score = []
        best_score = 0
        last_epoch = 0
        print("Training model from scratch")


    scaler = torch.cuda.amp.GradScaler()


    if (SHOW_VAL_INTERVAL > 0):
        display_check = np.arange(SHOW_VAL_INTERVAL-1, NUM_EPOCHS, SHOW_VAL_INTERVAL)
    else:
        display_check = 0



    for epoch in range(NUM_EPOCHS):
        print("\n----------------------------------------------------------------------------")
        print(f"\nepoch {epoch + 1}/{NUM_EPOCHS}")
        if LOAD_MODEL:
            print(f"total epoch {last_epoch+epoch + 1}/{last_epoch + NUM_EPOCHS}")

        # get loaders
        train_loader, val_loader = get_loaders(
            train_dataset,
            val_dataset,
            BATCH_SIZE,
            NUM_WORKERS,
            PIN_MEMORY,
        )

        # train model
        t_loss = train_fn(train_loader, model, optimizer, loss_fn, scaler)
        train_loss.append(t_loss)

        # evaluate model
        show_results = np.any(display_check == epoch)
        v_loss, f1_fib, f1_ax, dice = evaluate_fn(val_loader, model, loss_fn, device=DEVICE, show_results=show_results)
        val_loss.append(v_loss)
        f1_fibre.append(f1_fib)
        f1_axon.append(f1_ax)
        dice_score.append(dice)
        base_score = 0.4 * f1_fib + 0.4 * f1_ax + 0.2 * dice
        balanced_segmentation_score.append(base_score)

        print(
            "Train loss: {:.4f} | Val loss: {:.4f} | F1 fibre: {:.4f} | F1 axon: {:.4f} | Dice score: {:.4f} | Balanced Segmentation score: {:.4f}".format(
                t_loss, v_loss, f1_fib, f1_ax, dice, base_score
            )
        )

        # save best weights
        if base_score > best_score:
            best_score = base_score
            torch.save(model.state_dict(), "best_weights_model.pth")
            print("Saving best weights model")
            
        # save model checkpoint
        checkpoint = {
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch + 1 + last_epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "f1_fibre": f1_fibre,
            "f1_axon": f1_axon,
            "dice_score": dice_score,
            "balanced_segmentation_score": balanced_segmentation_score,
            "best_score": best_score,
        }
        save_checkpoint(checkpoint)

    torch.save(model.state_dict(), "last_epoch_model.pth")
    loss_plot_fn(train_loss, val_loss)
    loss_plot_log_fn(train_loss, val_loss)
    plot_segmentation_scores_fn(f1_fibre, f1_axon, dice_score, balanced_segmentation_score)

In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
# EXPORT BIOIMAGEIO MODEL

# import model
model = model_fn(DEVICE)

if(BIOIMAGEIO != "none"):
    # Load state_dict
    if(BIOIMAGEIO == "last"):
        model.load_state_dict(torch.load(r"last_epoch_model.pth"))
    elif(BIOIMAGEIO == "best"):
        model.load_state_dict(torch.load(r"best_weights_model.pth"))

    # export bioimageio model
    export_bioimageio(model, "AimSegDL", True, r'data_brightfield/test_images/A63A_t29.tif')